# 🌊 Volga River — Water Level Analysis (DAHITI)

Данные по уровню воды реки **Волга** из базы **DAHITI** (Database for Hydrological Time Series of Inland Waters, TU München), основанной на спутниковой альтиметрии.

**Перед запуском:**
1. Зарегистрируйся (бесплатно): https://dahiti.dgfi.tum.de/en/register/
2. Возьми свой API-ключ в профиле: https://dahiti.dgfi.tum.de/en/frequently-asked-questions/api-key/
3. Вставь ключ в переменную `API_KEY` ниже и запусти ячейки по порядку.

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

%matplotlib inline
sns.set_style('whitegrid')

## 1. Скачиваем данные из DAHITI API

In [2]:
# --- ВСТАВЬ СВОЙ КЛЮЧ ---
API_KEY = "PASTE_YOUR_API_KEY_HERE"
DAHITI_ID = 14685   # Volga River (id из ссылки dahiti.dgfi.tum.de/en/14685/...)

url = "https://dahiti.dgfi.tum.de/api/v2/download-water-level/"
params = {"api_key": API_KEY, "dahiti_id": DAHITI_ID, "format": "csv"}

resp = requests.get(url, params=params)
resp.raise_for_status()

# сохраняем локально
with open("volga_water_level.csv", "w") as f:
    f.write(resp.text)

print("Скачано символов:", len(resp.text))
print(resp.text[:300])   # смотрим первые строки файла

HTTPError: 403 Client Error: Forbidden for url: https://dahiti.dgfi.tum.de/api/v2/download-water-level/?api_key=PASTE_YOUR_API_KEY_HERE&dahiti_id=14685&format=csv

## 2. Загружаем в DataFrame и смотрим

In [ ]:
df = pd.read_csv("volga_water_level.csv")

# приводим дату к datetime (имя столбца может отличаться — смотри df.columns)
date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).reset_index(drop=True)

print(df.columns.tolist())
print(df.shape)
df.head()

## 3. График: уровень воды во времени

In [ ]:
# столбец с уровнем воды обычно называется 'water_level' или 'wse'
level_col = [c for c in df.columns if 'level' in c.lower() or c.lower() == 'wse'][0]

plt.figure(figsize=(14, 5))
plt.plot(df[date_col], df[level_col], color='royalblue', lw=1)
plt.title('Volga River — Water Surface Elevation over time')
plt.xlabel('Date')
plt.ylabel('Water level (m)')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Сезонность: средний уровень по месяцам

In [ ]:
df['Year'] = df[date_col].dt.year
df['Month'] = df[date_col].dt.month

monthly = df.groupby('Month')[level_col].mean()

plt.figure(figsize=(10, 5))
sns.barplot(x=monthly.index, y=monthly.values, hue=monthly.index,
            palette='viridis', legend=False)
plt.title('Средний уровень воды по месяцам (сезонность)')
plt.xlabel('Месяц')
plt.ylabel('Средний уровень (m)')
plt.show()